# Part 2: Time Series Modeling

In this notebook, you will implement functions to extract features from time series data and build ARIMA models.

In [2]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from pathlib import Path
import os

# Set style for plots
plt.style.use('seaborn-v0_8')
%matplotlib inline

## 1. Feature Extraction

Implement the `extract_time_series_features` function to calculate rolling window features.

In [3]:
def extract_time_series_features(data, window_size=60):
    """Extract rolling window features from time series data.
    
    Parameters
    ----------
    data : pd.DataFrame
        Preprocessed physiological data
    window_size : int
        Size of the rolling window in seconds
        
    Returns
    -------
    pd.DataFrame
        DataFrame containing extracted features for each signal
    """
    data = data.copy()
    data['timestamp'] = pd.to_datetime(data['timestamp'])
    data = data.set_index('timestamp').sort_index()  
    
    features = []

    def autocorr(x, lag=1):
        """Autocorrelation at lag 1."""
        if len(x) <= lag:
            return 0
        return x.autocorr(lag=lag)

    signals = ['heart_rate', 'eda', 'temperature']
    
    for signal in signals:
        if signal not in data.columns:
            continue

        rolling = data[signal].rolling(f'{window_size}s')

        stats_df = pd.DataFrame({
            f'{signal}_mean': rolling.mean(),
            f'{signal}_std': rolling.std(),
            f'{signal}_min': rolling.min(),
            f'{signal}_max': rolling.max(),
            f'{signal}_autocorr_lag1': rolling.apply(lambda x: autocorr(x, lag=1), raw=False)
        })

        features.append(stats_df)

    # Combine features from each signal
    feature_df = pd.concat(features, axis=1)
    feature_df = feature_df.dropna().reset_index()

    return feature_df

    pass

In [4]:
df_clean = pd.read_csv('data/processed/S1_processed.csv')
features_df = extract_time_series_features(df_clean, window_size=60)


/home/codespace/.local/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


## 2. ARIMA Modeling

Implement the `build_arima_model` function to fit ARIMA models and generate diagnostic plots.

In [5]:
def build_arima_model(series, order=(1,1,1), output_dir='plots'):
    """Fit an ARIMA model to the time series and generate diagnostic plots.
    
    Parameters
    ----------
    series : pd.Series
        Time series data to model
    order : tuple
        (p,d,q) order of the ARIMA model
    output_dir : str
        Directory to save diagnostic plots
        
    Returns
    -------
    statsmodels.tsa.arima.model.ARIMAResults
        Fitted ARIMA model
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Use signal name from the series if available
    signal_name = series.name if series.name else 'signal'
    base_name = f"{signal_name}_arima"

    # Fit ARIMA model
    model = ARIMA(series, order=order)
    model_fit = model.fit()

    # Plot observed vs fitted
    plt.figure(figsize=(10, 4))
    plt.plot(series.index, series, label='Observed', color='blue')
    plt.plot(series.index, model_fit.fittedvalues, label='Fitted', color='red')
    plt.title(f'ARIMA Fit: {signal_name}')
    plt.xlabel("Time")
    plt.ylabel(signal_name)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{base_name}_fit.png"))
    plt.close()

    # Plot residuals
    residuals = model_fit.resid
    plt.figure(figsize=(10, 4))
    plt.plot(residuals, label='Residuals', color='green')
    plt.axhline(0, color='black', linestyle='--')
    plt.title(f'ARIMA Residuals: {signal_name}')
    plt.xlabel("Time")
    plt.ylabel("Residuals")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{base_name}_residuals.png"))
    plt.close()

    return model_fit

    pass

In [6]:
series = df_clean['heart_rate']
fitted_model = build_arima_model(series, order=(1,1,1), output_dir='plots')
fitted_model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:             heart_rate   No. Observations:                45713
Model:                 ARIMA(1, 1, 1)   Log Likelihood              -75680.337
Date:                Wed, 30 Apr 2025   AIC                         151366.674
Time:                        23:38:56   BIC                         151392.864
Sample:                             0   HQIC                        151374.912
                              - 45713                                         
Covariance Type:                  opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.9368      0.003    356.887      0.000       0.932       0.942
ma.L1         -0.8726      0.003   -300.446      0.000      -0.878      -0.867
sigma2         1.6053      0.000   4408.038      0.000       1.605       1.606
===================================================================================
Ljung-Box (L1) (Q):                  71.78   Jarque-Bera (JB):       17284091717.63
Prob(Q):                              0.00   Prob(JB):                         0.00
Heteroskedasticity (H):               7.79   Skew:                            -1.50
Prob(H) (two-sided):                  0.00   Kurtosis:                      3015.41
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""